# 프로젝트 실습 — 축구 경기 분석: 탐지 · 분할 · 추적

컴퓨터비전 부트캠프 · 독립 프로젝트 모듈 · 4차시 (약 200분)

지난 fine-tuning 실습에서 우리는 사전학습 모델에 **새 어휘**를 가르쳤습니다.
이번에는 그 기술을 출발점 삼아, 사진 한 장이 아니라 **경기 영상 전체**를 분석하는
시스템을 만듭니다. 파이프라인은 세 단계로 자랍니다.

```
탐지 (Detection)      →  이 프레임의 어디에 무엇이 있는가          [박스]
분할 (Segmentation)   →  그 박스 안에서 정확히 어떤 픽셀인가        [마스크]
추적 (Tracking)       →  시간이 흐르면 그 객체가 어디로 가는가      [ID + 궤적]
```

| 차시 | 내용 | 결과물 |
|---|---|---|
| 1차시 | STEP 0~3 — 데이터 점검, 기준선, fine-tuning | 축구 전용 탐지 모델 (학습 걸어두고 종료) |
| 2차시 | STEP 4~5 — 성적표 읽기, SAM 분할 | 클래스별 AP 표, 선수 실루엣 마스크 |
| 3차시 | STEP 6~7 — ByteTrack 추적, 궤적·팀 구분 | ID·궤적·팀 색이 입혀진 분석 영상 |
| 4차시 | 팀 미션 — 자유 분석 | 우리 팀의 경기 분석 리포트 + 발표 |

> **준비물**: `soccer_clip.mp4` (추적용 클립). 데이터셋은 노트북이 알아서 받습니다.
> 네트워크가 불안한 교육장에서는 `football_players_part1~4.zip`을 먼저 업로드해 두세요.


---
# 1차시 — 탐지: 축구 전용 모델 만들기

## STEP 0 — 환경 준비

런타임 유형이 **GPU(T4)** 인지 먼저 확인합니다. (런타임 → 런타임 유형 변경 → T4 GPU)


In [ ]:
!nvidia-smi | head -12
%pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)

In [ ]:
# 한글 폰트 (그래프 라벨용) — 설치 후 이 셀만 다시 실행하면 됩니다
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.font_manager as fm, matplotlib.pyplot as plt
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf")
plt.rcParams["font.family"] = "NanumBarunGothic"
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 준비 완료")

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict

def show(*imgs, titles=None, w=7):
    """BGR 이미지들을 나란히 표시"""
    fig, axes = plt.subplots(1, len(imgs), figsize=(w*len(imgs), w*0.62))
    axes = [axes] if len(imgs) == 1 else axes
    for ax, im in zip(axes, imgs):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off")
    if titles:
        for ax, t in zip(axes, titles): ax.set_title(t, fontsize=11)
    plt.tight_layout(); plt.show()

CLASSES = ["ball", "goalkeeper", "player", "referee"]
CLASS_KO = {"ball": "공", "goalkeeper": "골키퍼", "player": "선수", "referee": "심판"}
COLORS = {0: (60,76,231), 1: (219,152,52), 2: (113,204,46), 3: (182,89,155)}  # BGR
print("준비 완료")

## STEP 1 — 데이터셋: 방송 화면 속의 네 가지 역할

**football-players-detection** (Roboflow, CC BY 4.0) — 축구 중계 방송 프레임에
`ball / goalkeeper / player / referee` 4개 클래스가 라벨링된 데이터셋입니다.

- 규모: train **612** / valid **38** / test **13** (원본 1920×1080)
- 특징: 지난 실습의 야생동물과 달리, 네 클래스가 전부 **"사람 또는 작은 물체"** 입니다.
  COCO 사전학습 모델의 눈에는 어떻게 보일까요? (STEP 2에서 확인)

아래 셀은 두 경로 중 하나로 데이터를 준비합니다 —
① 업로드된 백업 zip이 있으면 그것을 사용, ② 없으면 GitHub에서 원본을 받습니다.


In [ ]:
import os, glob
if glob.glob("football_players_part*.zip"):                      # ① 백업 zip 업로드됨
    print("백업 zip 발견 → 압축 해제 (1024px 축소판)")
    !mkdir -p dataset && for f in football_players_part*.zip; do unzip -q -o $f -d dataset; done
elif not os.path.isdir("dataset/train"):                          # ② GitHub에서 원본
    print("GitHub에서 데이터셋 다운로드 중 (~150MB, 1분 이내)")
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/Sam120204/Soccer_Analysis_ML_YOLOv5 _src
    !cd _src && git sparse-checkout set training/football-players-detection-1 -q
    !mv _src/training/football-players-detection-1/football-players-detection-1 dataset && rm -rf _src
print("완료")

In [ ]:
# 데이터가 제대로 왔는지 확인
for s in ["train", "valid", "test"]:
    n_img = len(list(Path(f"dataset/{s}/images").glob("*.jpg")))
    n_lbl = len(list(Path(f"dataset/{s}/labels").glob("*.txt")))
    print(f"{s:5s}: 이미지 {n_img:4d} 장 / 라벨 {n_lbl:4d} 개")
    assert n_img == n_lbl > 0, f"{s} 데이터가 비어 있거나 개수가 다릅니다!"
print("\n데이터셋 정상입니다.")

### 라벨 형식 복습 → TODO 1

라벨 한 줄 = `클래스번호 중심x 중심y 폭 높이` (모두 0~1 정규화) — 지난 실습과 같습니다.

**모델을 학습시키기 전, 데이터부터 의심하는 것**이 실무의 첫 습관입니다.
가장 기본적인 점검이 **클래스 분포**입니다: 어떤 클래스가 몇 번 등장하는가?

> **TODO 1** — train 라벨 파일 612개를 전부 읽어 클래스별 등장 횟수를 세고,
> 막대그래프로 그리세요. 힌트: 각 줄의 **첫 번째 숫자**가 클래스 번호이고,
> `counts[int(...)] += 1` 꼴로 세면 됩니다.


In [ ]:
# ★★★ TODO 1 ★★★  train 라벨의 클래스 분포 세기
counts = Counter()
for lf in Path("dataset/train/labels").glob("*.txt"):
    for line in lf.read_text().strip().split("\n"):
        if not line.strip():
            continue
        cls_id = None    # <-- ① 이 줄의 클래스 번호 (첫 번째 숫자를 int로)
        counts[cls_id] += 1

# 막대그래프
names_ko = [f"{CLASSES[i]}\n({CLASS_KO[CLASSES[i]]})" for i in range(4)]
values = [counts[i] for i in range(4)]
plt.figure(figsize=(7,4))
plt.bar(names_ko, values, color=["#e74c3c","#3498db","#2ecc71","#9b59b6"])
for i, v in enumerate(values):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=11)
plt.title("train 클래스 분포 — 무엇이 많이/적게 등장하나?"); plt.ylabel("인스턴스 수")
plt.show()

In [ ]:
# TODO 1 검사 셀 — 통과할 때까지 위 셀을 수정하세요
assert sum(counts.values()) > 10000, "라벨 줄 수가 너무 적습니다. 파싱을 확인하세요."
assert counts[2] > 10000, "player(2번)가 1만 개 이상이어야 합니다."
assert counts[0] < 600,   "ball(0번)은 600개 미만이어야 합니다."
print("통과! player", f"{counts[2]:,}개 vs ball {counts[0]:,}개 —",
      "공은 선수의 1/20도 안 됩니다. 이 불균형을 기억해 두세요 (STEP 4에서 다시 만납니다).")

In [ ]:
# 정답(GT) 라벨을 눈으로 확인 — 4클래스를 색으로 구분
def draw_gt(img_path):
    img = cv2.imread(str(img_path)); H, W = img.shape[:2]
    lbl = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")
    for line in lbl.read_text().strip().split("\n"):
        c, cx, cy, w, h = map(float, line.split()); c = int(c)
        x1, y1 = int((cx-w/2)*W), int((cy-h/2)*H); x2, y2 = int((cx+w/2)*W), int((cy+h/2)*H)
        cv2.rectangle(img, (x1,y1), (x2,y2), COLORS[c], 2)
        cv2.putText(img, CLASSES[c], (x1, y1-4), 0, 0.6, COLORS[c], 2)
    return img

samples = sorted(Path("dataset/valid/images").glob("*.jpg"))[:2]
show(*[draw_gt(p) for p in samples], titles=["GT 예시 1", "GT 예시 2"], w=9)
print("빨강=공, 파랑=골키퍼, 초록=선수, 보라=심판")

## STEP 2 — 학습 전 기준선: COCO의 눈으로 본 축구장

fine-tuning의 가치는 **학습 전 모델이 무엇을 못 하는지** 기록해야 증명됩니다.
COCO 사전학습 `yolo11n`으로 검증 이미지를 추론해 봅시다. COCO에는 `person`과
`sports ball`은 있지만 **goalkeeper, referee라는 어휘는 없습니다.**


In [ ]:
from ultralytics import YOLO
coco_model = YOLO("yolo11n.pt")     # COCO 80클래스 사전학습

for p in samples:
    r = coco_model(str(p), conf=0.25, verbose=False)[0]
    det = Counter(r.names[int(c)] for c in r.boxes.cls)
    print(f"{p.name[:30]:32s} → {dict(det)}")
show(*[YOLO("yolo11n.pt")(str(p), conf=0.25, verbose=False)[0].plot() for p in samples],
     titles=["COCO 모델의 눈 1", "COCO 모델의 눈 2"], w=9)

### 관찰 정리

| 우리가 원하는 것 | COCO 모델이 하는 것 |
|---|---|
| 선수 / 골키퍼 / 심판 구분 | 전부 똑같은 `person` |
| 공 위치 (작아도) | `sports ball`을 가끔, 대부분 놓침 |
| 축구장 문맥 이해 | 관중석의 사람들까지 `person`으로 |

지난 실습과 똑같은 결론입니다 — **모델은 아는 단어로밖에 세상을 부르지 못합니다.**
threshold를 조절해도 `person`이 골키퍼가 되지는 않습니다. 어휘를 바꾸려면 fine-tuning입니다.


## STEP 3 — Fine-tuning: 역할 어휘 가르치기

### TODO 2 — data.yaml 작성 (지난 실습 복습)

학습 설명서를 직접 씁니다. 경로 2개와 클래스 이름 4개를 채우세요.
**클래스 순서는 라벨 번호 순서(0=ball, 1=goalkeeper, 2=player, 3=referee)와
정확히 같아야 합니다** — STEP 1의 막대그래프 순서가 그대로 답입니다.


In [ ]:
%%writefile dataset/data.yaml
# ★★★ TODO 2 ★★★  아래 빈칸을 채우세요
path: /content/dataset
train: # <-- ① train 이미지 폴더 (path 기준 상대경로)
val:   # <-- ② 검증 이미지 폴더
nc: 4
names: []   # <-- ③ 클래스 이름 4개, 번호 순서대로

In [ ]:
# TODO 2 검사 셀 — '통과!'가 나오기 전에는 다음으로 넘어가지 마세요
import yaml
cfg = yaml.safe_load(Path("dataset/data.yaml").read_text())
assert cfg.get("nc") == 4, "nc는 4여야 합니다"
assert [s.lower() for s in cfg.get("names", [])] == CLASSES, f"names 순서가 다릅니다: {cfg.get('names')}"
for k in ["train", "val"]:
    p = Path(cfg["path"]) / cfg[k]
    assert p.is_dir() and len(list(p.glob('*.jpg'))) > 0, f"{k} 경로가 비었거나 없습니다: {p}"
print("통과! 학습 설명서 완성 —", cfg["names"])

### TODO 3 — 학습 실행

| 인자 | 값 | 왜? |
|---|---|---|
| `data` | `"dataset/data.yaml"` | 방금 쓴 설명서 |
| `epochs` | `EPOCHS` (25) | T4에서 약 10~13분 |
| `imgsz` | `960` | **공이 작아서** 지난 실습(640)보다 크게 봅니다 |

방송 화면에서 공은 10픽셀 남짓입니다. 640으로 줄이면 공이 4~5픽셀이 되어
거의 안 보입니다 — 해상도는 작은 객체 성능의 첫 번째 레버입니다.

셀을 실행했으면 **로그가 흐르는 동안 아래 '읽을거리'로 내려가세요.** (1차시는 여기까지)


In [ ]:
# ★★★ TODO 3 ★★★  빈칸 3개를 채워 학습을 시작하세요
from ultralytics import YOLO
EPOCHS = 25
model = YOLO("yolo11n.pt")
results = model.train(
    data=None,      # <-- ① 학습 설명서 경로 (문자열)
    epochs=None,    # <-- ② 반복 횟수 (위에서 정한 변수)
    imgsz=None,     # <-- ③ 입력 크기 (공이 작다 → 960)
    batch=16,
    project="runs", name="soccer", exist_ok=True,
)

### 학습을 기다리는 12분 동안 — 읽을거리

**① 로그에서 볼 것** — `Overriding model.yaml nc=80 with nc=4` (head 교체 증거),
box/cls 손실이 내려가는 추세, epoch마다 갱신되는 mAP50. 지난 실습과 같은 문법입니다.

**② 다음 차시 예고 — SAM (Segment Anything Model)**
박스는 "여기쯤"이고 마스크는 "정확히 이 픽셀들"입니다. SAM은 1,100만 장으로
사전학습된 분할 파운데이션 모델로, **클릭 한 번이나 박스 하나를 프롬프트로 주면**
그 객체의 픽셀을 오려냅니다. 우리 탐지기가 조준(박스)하고 SAM이 오려내는(마스크)
분업 — 라벨 없이 분할을 얻는 실전 패턴입니다.

**③ ByteTrack — 3차시 예고**
추적은 "매 프레임의 탐지 결과를 시간으로 꿰는 일"입니다. ByteTrack의 아이디어는
단순합니다: 확신 높은 박스로 먼저 잇고, **확신 낮은 박스도 버리지 않고** 2차로
잇는다 — 가려진 선수가 흐릿하게 잡혀도 ID가 끊기지 않는 이유입니다.

---
**1차시 끝.** 학습이 끝났다면 `runs/soccer/weights/best.pt`가 생겼는지 확인하고,
Colab 세션이 끊길 것 같으면 best.pt를 다운로드해 두세요.


---
# 2차시 — 성적표 읽기, 그리고 박스에서 픽셀로

## STEP 4 — 결과 분석

먼저 학습 곡선과 혼동 행렬을 봅니다. (세션이 초기화됐다면 STEP 0의 설치 셀과
import 셀만 다시 실행 후 `model = YOLO("runs/soccer/weights/best.pt")`로 이어가세요.)


In [ ]:
from IPython.display import Image as IPyImage, display
display(IPyImage("runs/soccer/results.png", width=900))
display(IPyImage("runs/soccer/confusion_matrix_normalized.png", width=560))

### TODO 4 — 클래스별 성적 꺼내기

전체 평균 mAP는 요약일 뿐, **클래스별 AP**가 진짜 이야기를 합니다.
`model.val()`이 돌려주는 metrics 객체에서:

- `m.box.map50` — 전체 mAP@0.5, `m.box.map` — mAP@0.5:0.95
- `m.box.maps` — **클래스별** AP@0.5:0.95 배열 (인덱스 = 클래스 번호)

> STEP 1에서 본 클래스 불균형(공 519 vs 선수 12,228)이 성적에 어떻게 나타났을까요?


In [ ]:
# ★★★ TODO 4 ★★★  검증 성적을 클래스별로 뜯어보세요
model = YOLO("runs/soccer/weights/best.pt")
m = model.val(data="dataset/data.yaml", verbose=False)

overall_map50 = None   # <-- ① 전체 mAP@0.5
overall_map   = None   # <-- ② 전체 mAP@0.5:0.95
per_class     = None   # <-- ③ 클래스별 AP 배열 (m.box.maps)

print(f"전체 mAP@0.5      : {overall_map50:.3f}")
print(f"전체 mAP@0.5:0.95 : {overall_map:.3f}\n")
for i, ap in enumerate(per_class):
    bar = "█" * int(ap * 40)
    print(f"{CLASSES[i]:12s} {CLASS_KO[CLASSES[i]]:4s} AP {ap:.3f} {bar}")

In [ ]:
# TODO 4 검사 셀
assert overall_map50 is not None and 0 < overall_map50 <= 1
assert per_class is not None and len(per_class) == 4
assert per_class[0] < per_class[2], "보통 ball AP가 player AP보다 낮게 나옵니다 — 값을 다시 확인하세요"
print(f"통과! 가장 어려운 클래스: {CLASSES[int(np.argmin(per_class))]}")

### 공은 왜 꼴찌인가 — 작은 객체 문제

원인은 세 겹입니다: **① 작다** (960에서도 몇 픽셀), **② 드물다** (선수의 1/20),
**③ 빠르다** (모션 블러). 실무의 처방전:

- 해상도 ↑ (`imgsz=1280`) — 우리가 이미 쓴 레버, 비용도 같이 오름
- **타일링(SAHI)** — 이미지를 조각내 확대 추론
- **공 전용 모델** — 실제 프로 축구 분석 파이프라인이 쓰는 방법 (선수 모델 + 공 모델 분리)

> 낮은 숫자는 실패가 아니라 **문제의 구조가 드러난 것**입니다. 4차시 미션에서
> 이 처방전 중 하나를 시도해 봐도 좋습니다.


In [ ]:
# 학습 전 vs 학습 후 — 같은 이미지, 다른 어휘
for p in samples[:2]:
    before = YOLO("yolo11n.pt")(str(p), conf=0.25, verbose=False)[0].plot()
    after  = model(str(p), conf=0.25, verbose=False)[0].plot()
    show(before, after, titles=["학습 전 — 전부 person", "학습 후 — 역할이 보인다"], w=9)

## STEP 5 — 분할: SAM으로 박스를 마스크로

**SAM(Segment Anything Model)** 은 "무엇이든 오려내는" 분할 파운데이션 모델입니다.
클래스를 모르지만(이름 없음), **프롬프트(점·박스)가 가리키는 것"을 픽셀 단위로
오려내는 능력**이 있습니다. 우리의 분업 구조:

```
우리 탐지기(best.pt)  →  "선수가 여기 있다" (박스 + 클래스)
SAM (mobile_sam)      →  "그렇다면 정확히 이 픽셀들이다" (마스크)
```

라벨 한 장 없이 분할을 얻습니다. 산업 현장에서 **탐지 데이터셋 → 분할 데이터셋
자동 변환**(auto-labeling)에 그대로 쓰이는 패턴입니다.


In [ ]:
from ultralytics import SAM
sam = SAM("mobile_sam.pt")      # 40MB 경량 SAM — 자동 다운로드
sam.info()

### TODO 5 — 탐지 → 분할 파이프라인 연결

① 우리 모델로 이미지를 추론해 **player 박스만** 골라내고 (`r.boxes.cls == 2`),
② 그 박스들을 SAM에 프롬프트로 넘기세요: `sam(이미지경로, bboxes=박스배열)`.


In [ ]:
# ★★★ TODO 5 ★★★  탐지 박스를 SAM 프롬프트로
img_path = str(samples[0])

# ① 우리 탐지기로 추론
r = model(img_path, conf=0.4, verbose=False)[0]
is_player = r.boxes.cls.cpu().numpy() == 2          # player 클래스만
player_boxes = None   # <-- ① player 박스 좌표 (r.boxes.xyxy를 .cpu().numpy()로, is_player로 골라내기)

# ② SAM에 박스 프롬프트 전달
sam_res = None        # <-- ② sam(...) 호출: img_path와 bboxes=player_boxes

masks = sam_res[0].masks.data.cpu().numpy()
print(f"선수 {len(player_boxes)}명 → 마스크 {masks.shape[0]}개, 해상도 {masks.shape[1:]}")

In [ ]:
# 마스크 오버레이 + 유니폼 색 추출 (제공)
img = cv2.imread(img_path)
overlay = img.copy()
np.random.seed(0)
jersey_colors = []
for mask in masks:
    mb = mask.astype(bool)
    color = tuple(int(c) for c in np.random.randint(60, 255, 3))
    overlay[mb] = 0.45 * overlay[mb] + 0.55 * np.array(color)
    # 유니폼 색: 마스크 상반신 픽셀 중 '잔디색이 아닌' 픽셀의 평균
    ys, xs = np.where(mb)
    upper = ys < ys.min() + (ys.max() - ys.min()) * 0.5
    px = img[ys[upper], xs[upper]].astype(float)
    not_green = ~((px[:,1] > px[:,0]) & (px[:,1] > px[:,2]))   # G가 최대인 픽셀 제외
    jersey_colors.append(px[not_green].mean(axis=0) if not_green.any() else px.mean(axis=0))
show(img, overlay, titles=["원본", f"SAM 마스크 {len(masks)}개 — 박스가 실루엣이 됐다"], w=9)

# 색 스와치
sw = np.zeros((60, 60*len(jersey_colors), 3), np.uint8)
for i, c in enumerate(jersey_colors):
    sw[:, i*60:(i+1)*60] = np.array(c, np.uint8)
show(sw, titles=["선수별 유니폼 색 — 3차시 팀 구분의 재료"], w=9)

### 심화 상자 — 분할 '데이터셋'이 필요하다면

방금 한 일을 데이터셋 전체에 반복하면 **탐지 라벨 → 분할 라벨 자동 변환**입니다.
ultralytics는 이를 함수 하나로 제공합니다:

```python
from ultralytics.data.annotator import auto_annotate
auto_annotate(data="dataset/valid/images", det_model="runs/soccer/weights/best.pt",
              sam_model="mobile_sam.pt", output_dir="seg_labels")
```

생성된 라벨로 `yolo11n-seg.pt`를 fine-tuning하면 SAM 없이도 실시간 분할이 되는
전용 모델을 얻습니다 — 4차시 미션 후보 (★★★).

---
**2차시 끝.** 우리는 이제 프레임 한 장을 픽셀 수준까지 읽습니다. 다음 차시에 시간을 붙입니다.


---
# 3차시 — 추적: 시간을 꿰는 실

## STEP 6 — ByteTrack으로 ID 붙이기

탐지는 매 프레임 "선수 22명"이라고 말할 뿐, **7번 선수가 어디로 갔는지**는 모릅니다.
추적기는 프레임 간 박스를 이어 **ID를 유지**합니다. ultralytics에는 두 추적기가
내장되어 있고, 호출은 `model.track()` 하나입니다.

| tracker | 특징 |
|---|---|
| `bytetrack.yaml` | 가볍고 빠름. 낮은 confidence 박스도 2차 매칭에 활용 |
| `botsort.yaml` | 기본값. 카메라 움직임 보정 + 외형(ReID) 옵션 |

**`persist=True`** 가 핵심입니다 — 프레임을 하나씩 루프로 넣을 때 "이전 프레임의
추적 상태를 기억하라"는 뜻입니다. 이게 없으면 매 프레임 ID가 리셋됩니다.


In [ ]:
# 추적용 클립 준비 — soccer_clip.mp4를 왼쪽 파일 탭에 업로드하세요
import os
if not os.path.exists("soccer_clip.mp4"):
    print("업로드된 클립이 없어 예비 URL에서 받습니다 (30초 원본)")
    !curl -sL -o soccer_clip.mp4 https://raw.githubusercontent.com/Cshiva773/football-analysis/main/input_videos/08fd33_4.mp4
cap = cv2.VideoCapture("soccer_clip.mp4")
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS)
W, H = int(cap.get(3)), int(cap.get(4)); cap.release()
print(f"클립: {W}x{H}, {fps:.0f}fps, {n_frames}프레임 ({n_frames/fps:.0f}초)")

### TODO 6 — 추적 루프 완성

프레임 루프 안에서 `model.track()`을 호출하세요. 인자 세 개:
**프레임, `persist=True`, `tracker="bytetrack.yaml"`** (+ `conf=0.25, verbose=False`).


In [ ]:
# ★★★ TODO 6 ★★★  추적 실행 → ID 달린 영상 만들기
cap = cv2.VideoCapture("soccer_clip.mp4")
out = cv2.VideoWriter("track_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

while True:
    ok, frame = cap.read()
    if not ok: break
    r = None    # <-- model.track( 프레임, persist=?, tracker=?, conf=0.25, verbose=False )[0]
    out.write(r.plot())          # 박스+ID가 그려진 프레임
cap.release(); out.release()
print("track_raw.mp4 완성")

In [ ]:
# Colab에서 재생 (브라우저용 코덱으로 변환 후 표시)
from IPython.display import HTML
from base64 import b64encode
!ffmpeg -y -v error -i track_raw.mp4 -c:v libx264 -pix_fmt yuv420p track_raw_h264.mp4
HTML(f'<video width="880" controls src="data:video/mp4;base64,{b64encode(open("track_raw_h264.mp4","rb").read()).decode()}"></video>')

## STEP 7 — 궤적과 팀: 추적을 '분석'으로 바꾸기

ID가 생기면 할 수 있는 일이 폭발합니다. 두 가지를 만들어 봅니다.

1. **궤적** — ID별 발끝 좌표를 누적해 폴리라인으로 그리기 (**TODO 7**)
2. **팀 구분** — 유니폼 색(STEP 5의 재료!)을 KMeans로 2팀 클러스터링 (제공)


In [ ]:
# 팀 구분 도우미 (제공) — 셔츠 크롭에서 잔디색을 뺀 평균색 → KMeans 2팀
from sklearn.cluster import KMeans

def shirt_color(frame, box):
    """박스 상단 절반(셔츠 영역)에서 잔디색 제외 평균 BGR"""
    x, y, w, h = box
    x1, y1 = int(x - w/2), int(y - h/2)
    crop = frame[max(y1,0):int(y1 + h/2), max(x1,0):int(x1 + w)]
    if crop.size == 0: return np.array([0., 0, 0])
    px = crop.reshape(-1, 3).astype(float)
    not_green = ~((px[:,1] > px[:,0]) & (px[:,1] > px[:,2]))
    return px[not_green].mean(axis=0) if not_green.sum() > 10 else px.mean(axis=0)

def fit_teams(video, model, n_probe=30):
    """앞쪽 프레임들에서 선수 셔츠색을 모아 2팀 KMeans 학습"""
    cap = cv2.VideoCapture(video); feats = []
    for _ in range(n_probe):
        ok, frame = cap.read()
        if not ok: break
        r = model(frame, conf=0.4, verbose=False)[0]
        for box, c in zip(r.boxes.xywh.cpu().numpy(), r.boxes.cls.cpu().numpy()):
            if int(c) == 2: feats.append(shirt_color(frame, box))
    cap.release()
    km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(feats)
    return km

km = fit_teams("soccer_clip.mp4", model)
TEAM_BGR = [tuple(int(v) for v in c) for c in km.cluster_centers_]
sw = np.zeros((60, 240, 3), np.uint8); sw[:, :120] = TEAM_BGR[0]; sw[:, 120:] = TEAM_BGR[1]
show(sw, titles=["KMeans가 찾아낸 두 팀의 색"], w=5)

In [ ]:
# ★★★ TODO 7 ★★★  궤적 누적 + 팀 색 렌더링
model = YOLO("runs/soccer/weights/best.pt")   # 새로 로드 → 추적 ID가 1부터 시작
track_history = defaultdict(list)     # ID → [(x, y_발끝), ...]
team_votes    = defaultdict(list)     # ID → [팀번호 투표, ...]

cap = cv2.VideoCapture("soccer_clip.mp4")
out = cv2.VideoWriter("track_teams.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
while True:
    ok, frame = cap.read()
    if not ok: break
    r = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=0.25, verbose=False)[0]
    canvas = frame.copy()
    if r.boxes.id is not None:
        for box, tid, c in zip(r.boxes.xywh.cpu().numpy(),
                               r.boxes.id.int().cpu().tolist(),
                               r.boxes.cls.cpu().numpy()):
            if int(c) != 2: continue                    # 선수만
            x, y, w, h = box
            team = int(km.predict([shirt_color(frame, box)])[0]); team_votes[tid].append(team)
            color = TEAM_BGR[round(np.mean(team_votes[tid]))]

            # <-- ① 발끝 좌표 (x, y + h/2) 를 track_history[tid]에 추가하세요

            pts = np.array(track_history[tid][-45:], np.int32).reshape(-1, 1, 2)
            # <-- ② cv2.polylines(...)로 canvas에 궤적을 그리세요 (색=color, 두께 2)

            cv2.rectangle(canvas, (int(x-w/2), int(y-h/2)), (int(x+w/2), int(y+h/2)), color, 2)
            cv2.putText(canvas, str(tid), (int(x-w/2), int(y-h/2)-4), 0, 0.55, color, 2)
    out.write(canvas)
cap.release(); out.release()
print("track_teams.mp4 완성 —", len(track_history), "명 추적")

In [ ]:
!ffmpeg -y -v error -i track_teams.mp4 -c:v libx264 -pix_fmt yuv420p track_teams_h264.mp4
HTML(f'<video width="880" controls src="data:video/mp4;base64,{b64encode(open("track_teams_h264.mp4","rb").read()).decode()}"></video>')

In [ ]:
# 보너스 (제공) — 추적 데이터는 이미 '표'다: 이동거리 랭킹 + 위치 히트맵
dist = {tid: sum(np.hypot(x2-x1, y2-y1) for (x1,y1),(x2,y2) in zip(h, h[1:]))
        for tid, h in track_history.items() if len(h) > int(fps*3)}
top = sorted(dist.items(), key=lambda kv: -kv[1])[:10]
print("이동거리 TOP 10 (픽셀):")
for tid, d in top:
    team = round(np.mean(team_votes[tid]))
    print(f"  ID {tid:3d} (팀 {team})  {d:7.0f}px  {'▮'*int(d/top[0][1]*30)}")

pts = np.array([p for h in track_history.values() for p in h])
plt.figure(figsize=(9, 5))
plt.hist2d(pts[:,0], pts[:,1], bins=[48, 27], cmap="hot")
plt.gca().invert_yaxis(); plt.colorbar(label="방문 빈도")
plt.title("선수 위치 히트맵 — 경기가 어디서 이루어졌나"); plt.show()

### 주의: 지금 거리는 '픽셀' 거리다

카메라 원근 때문에 화면 위쪽 1픽셀과 아래쪽 1픽셀은 실제 거리가 다릅니다.
미터 단위로 바꾸려면 **원근 변환(homography)** 으로 경기장 좌표계에 투영해야
합니다 — 프로 분석 파이프라인의 다음 단계이자, 미션 메뉴의 ★★★ 항목입니다.

---
**3차시 끝.** 탐지 → 분할 → 추적, 파이프라인이 완성됐습니다. 이제 여러분 차례입니다.


---
# 4차시 — 팀 미션: 우리 팀의 경기 분석 리포트

지금까지 만든 **빌딩블록**으로, 팀마다 분석 주제 1개를 골라 구현하고
**3~5분 발표**로 마무리합니다.

## 빌딩블록 인벤토리 (여러분이 이미 가진 것)

| 재료 | 변수/함수 | 쓸모 |
|---|---|---|
| 축구 전용 탐지기 | `model` (best.pt) | 프레임 → 박스+클래스 |
| 분할 | `sam`, `bboxes=` 프롬프트 | 박스 → 픽셀 마스크 |
| 추적 | `model.track(persist=True)` | ID 유지 |
| 궤적 | `track_history` | ID → 좌표 시계열 |
| 팀 색 | `km`, `shirt_color()`, `team_votes` | 선수 → 팀 |

## 미션 메뉴 (하나를 고르거나 직접 제안)

| 난이도 | 미션 | 힌트 |
|---|---|---|
| ★ | **팀별 히트맵 비교** — 두 팀의 점유 영역은 어떻게 다른가? | 히트맵 셀을 `team_votes`로 팀별 분리 |
| ★ | **활동량 랭킹** — 프레임당 이동거리로 속도 추정, 최다 활동 선수는? | `dist` 계산에 시간(프레임 수) 나누기 |
| ★★ | **공 점유율** — 매 프레임 공과 가장 가까운 선수의 팀 집계 | 공(클래스 0) 탐지 + 최근접 선수 거리 |
| ★★ | **골키퍼 행동반경** — GK(클래스 1)의 궤적만 따로 시각화 | `if int(c) != 2` 조건을 바꾸기 |
| ★★★ | **패스 감지 근사** — 공 최근접 선수가 A→B로 바뀌는 순간 잡기 | 점유 선수 시계열에서 전환점 탐지 |
| ★★★ | **분할 전용 모델** — auto_annotate로 seg 라벨 생성 → yolo11n-seg 학습 | STEP 5 심화 상자 |
| ★★★ | **내 영상에 적용** — 직접 구한 다른 경기/스포츠 영상으로 전체 파이프라인 | 도메인 갭을 몸으로 확인 |

## 제출물과 평가

- 제출: 이 노트북 사본 (미션 셀 + 결과 시각화 포함) / 발표: 3~5분, "질문 → 방법 → 결과 → 한계"
- 평가 루브릭 (100점):

| 항목 | 배점 | 기준 |
|---|---|---|
| 구현 완성도 | 40 | 파이프라인이 끝까지 돌아가고 결과물이 재현되는가 |
| 분석의 타당성 | 30 | 질문이 명확하고, 방법이 질문에 맞으며, **한계를 스스로 아는가** |
| 시각화 전달력 | 20 | 그림만 보고도 결론이 읽히는가 (제목·범례·단위) |
| 발표 | 10 | 시간 준수, 역할 분담, 질의응답 |


---
## 마무리 — 오늘 만든 것

```
사진 한 장            영상 전체
   │                     │
   ▼                     ▼
[탐지] ──박스──▶ [분할] ──마스크──▶ [추적] ──ID·궤적──▶ [분석]
 fine-tuning      SAM 프롬프트       ByteTrack          여러분의 질문
```

한 문장 요약 — **모델은 부품이고, 시스템은 질문이 만든다.**

### 출처·라이선스
- 데이터셋: [football-players-detection](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc) (Roboflow Universe, **CC BY 4.0**)
- 클립: DFL Bundesliga Data Shootout 공개 클립 (교육 목적 사용)
- 도구: [Ultralytics YOLO11](https://docs.ultralytics.com) · [SAM](https://docs.ultralytics.com/models/sam/) · [ByteTrack](https://docs.ultralytics.com/modes/track/)
